# 🚗 ParkSmart City Services — Round 2: EDA & Feature Engineering

**Organization:** ParkSmart City Services | **Sector:** Smart City  
**Assessment:** Use Case 17 — Parking Utilization Insight  
**Date:** October 2026

---

## Business Problem

ParkSmart City Services operates a network of urban parking zones across the city.
Management needs evidence-based answers to six operational questions:

| # | Business Question |
|---|-------------------|
| 1 | Which zones have the **highest occupancy**? |
| 2 | **When** are peak parking periods? |
| 3 | Which zones have **low turnover**? |
| 4 | How do **events** affect occupancy? |
| 5 | Does **rain** change parking behaviour? |
| 6 | How does **parking duration** vary by zone type? |

This notebook performs a complete EDA and feature engineering workflow to answer those questions using five available datasets.

## 1  Environment Setup

In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 150, 'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27', 'axes.edgecolor': '#3a3d4d',
    'axes.labelcolor': '#e0e0e0', 'axes.titlecolor': '#ffffff',
    'text.color': '#e0e0e0', 'xtick.color': '#b0b0b0', 'ytick.color': '#b0b0b0',
    'grid.color': '#2a2d3d', 'grid.linestyle': '--', 'grid.alpha': 0.5,
    'legend.facecolor': '#1a1d27', 'legend.edgecolor': '#3a3d4d',
    'legend.labelcolor': '#e0e0e0', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 14, 'axes.labelsize': 11,
})
PALETTE = ['#6c63ff','#ff6584','#43e97b','#f7971e','#4facfe','#f093fb']
sns.set_theme(style='dark', rc={'axes.facecolor':'#1a1d27','figure.facecolor':'#0f1117'})

current_dir = Path('.').resolve()
if current_dir.name == 'notebooks':
    PROJECT_ROOT = current_dir.parent
elif (current_dir / 'notebooks').exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

SOURCE_DIR   = PROJECT_ROOT.parent / 'ParkSmart_Round2_Practice_Dataset'
DATA_PROC    = PROJECT_ROOT / 'data' / 'processed'
CHARTS_DIR   = PROJECT_ROOT / 'outputs' / 'charts'
REPORTS_DIR  = PROJECT_ROOT / 'outputs' / 'reports'

for d in [DATA_PROC, CHARTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def save_chart(fig, name):
    path = CHARTS_DIR / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'  ✓ Saved {path.name}')

def assign_time_period(hour):
    if pd.isna(hour):   return 'Unknown'
    if 0 <= hour < 6:   return 'Night'
    if 6 <= hour < 12:  return 'Morning'
    if 12 <= hour < 18: return 'Afternoon'
    return 'Evening'

def assign_util_cat(rate):
    if pd.isna(rate): return 'Unknown'
    if rate < 1.0:    return 'Low'
    if rate <= 4.0:   return 'Moderate'
    return 'High'

print('Project root :', PROJECT_ROOT)
print('Source data  :', SOURCE_DIR)
print('Python       :', sys.version)


Project root : /Users/manishbasnet/Desktop/DS SAT ROUND 2/parking-utilization
Source data  : /Users/manishbasnet/Desktop/DS SAT ROUND 2/ParkSmart_Round2_Practice_Dataset
Python       : 3.14.8 (main, Sep 30 2026, 17:55:09) [Clang 21.0.0 (clang-2100.3.34.2)]


## 2  Dataset Overview

In [2]:
def load_csv(name):
    path = SOURCE_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}')
    return pd.read_csv(path)

raw_sessions = load_csv('parking_sessions.csv')
raw_zones    = load_csv('zones.csv')
raw_payments = load_csv('payments.csv')
raw_calendar = load_csv('calendar.csv')
raw_weather  = load_csv('weather.csv')

for name, df_tmp in [('parking_sessions', raw_sessions), ('zones', raw_zones),
                      ('payments', raw_payments), ('calendar', raw_calendar),
                      ('weather', raw_weather)]:
    print(f'  {name:20s}: {df_tmp.shape[0]:,} rows x {df_tmp.shape[1]} cols')

  parking_sessions    : 1,212 rows x 8 cols
  zones               : 6 rows x 4 cols
  payments            : 1,205 rows x 4 cols
  calendar            : 60 rows x 4 cols
  weather             : 60 rows x 3 cols


In [3]:
print('=== parking_sessions — first 5 rows ===')
display(raw_sessions.head())
print('\nDtypes & nulls:')
display(raw_sessions.dtypes.to_frame('dtype').join(raw_sessions.isnull().sum().to_frame('nulls')))

=== parking_sessions — first 5 rows ===


,session_id,zone_id,zone,entry,exit,duration,fee,payment_id
0,S00001,Z03,Z03,2026-08-11 08:04:00,2026-08-11 07:34:00,52.0,26.00,P00001
1,S00002,Z06,Z06,2026-08-31 10:02:00,2026-08-31 13:26:00,204.0,119.00,P00002
2,S00003,Z05,Z05,2026-09-20 16:14:00,2026-09-20 17:41:00,87.0,43.50,P00003
3,S00004,Z05,Z05,2026-08-30 07:24:00,2026-08-30 11:16:00,232.0,116.00,P00004
4,S00005,Z03,Z03,2026-08-02 10:21:00,2026-08-02 09:51:00,235.0,97.92,P00005



Dtypes & nulls:


,dtype,nulls
session_id,str,0
zone_id,str,0
zone,str,0
entry,str,0
exit,str,23
duration,float64,10
fee,float64,0
payment_id,str,0


In [4]:
print('=== zones ==='); display(raw_zones)
print('\n=== calendar (first 5) ==='); display(raw_calendar.head())
print('\n=== weather (first 5) ===');  display(raw_weather.head())
print('\n=== payments (first 5) ==='); display(raw_payments.head())

=== zones ===


,zone_id,location,capacity,zone_type
0,Z01,Central Plaza,120,Commercial
1,Z02,City Hospital,85,Hospital
2,Z03,Riverfront,74,Recreational
3,Z04,Railway Station,150,Transit
4,Z05,Market Street,95,Commercial
5,Z06,University,110,Educational



=== calendar (first 5) ===


,date,weekday,holiday,event
0,2026-08-01,Saturday,1,0
1,2026-08-02,Sunday,1,0
2,2026-08-03,Monday,0,0
3,2026-08-04,Tuesday,0,0
4,2026-08-05,Wednesday,0,0



=== weather (first 5) ===


,date,rain,temperature
0,2026-08-01,1,29.4
1,2026-08-02,0,32.9
2,2026-08-03,0,23.3
3,2026-08-04,0,27.8
4,2026-08-05,0,31.4



=== payments (first 5) ===


,payment_id,session_id,payment_mode,amount
0,P00001,S00001,Card,26.00
1,P00002,S00002,UPI,119.00
2,P00003,S00003,Cash,43.50
3,P00004,S00004,UPI,116.00
4,P00005,S00005,UPI,97.92


In [5]:
display(raw_sessions.describe(include='all'))

,session_id,zone_id,zone,entry,exit,duration,fee,payment_id
count,1212,1212,1212,1212,1189,1202.000000,1212.000000,1212
unique,1200,6,10,1189,1171,NaN,NaN,1200
top,S00387,Z04,Z04,2026-09-13 06:21:00,2026-09-05 19:48:00,NaN,NaN,P00387
freq,2,255,251,2,2,NaN,NaN,2
mean,NaN,NaN,NaN,NaN,NaN,147.911814,67.798960,NaN
std,NaN,NaN,NaN,NaN,NaN,107.216577,51.902914,NaN
min,NaN,NaN,NaN,NaN,NaN,13.000000,10.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,73.000000,32.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,120.000000,52.500000,NaN
75%,NaN,NaN,NaN,NaN,NaN,188.000000,88.752500,NaN


### Dataset Purposes

| Dataset | Purpose | Key Columns |
|---------|---------|-------------|
| `parking_sessions` | One row per parking session | `session_id`, `zone_id`, `entry`, `exit`, `duration`, `fee`, `payment_id` |
| `zones` | Zone master — capacity & type | `zone_id`, `location`, `capacity`, `zone_type` |
| `payments` | Payment records | `payment_id`, `session_id`, `payment_mode`, `amount` |
| `calendar` | Daily holiday/event flags | `date`, `weekday`, `holiday`, `event` |
| `weather` | Daily weather | `date`, `rain`, `temperature` |

## 3  Data Quality Audit

In [6]:
sess_tmp = raw_sessions.copy()
sess_tmp['entry_ts'] = pd.to_datetime(sess_tmp['entry'], errors='coerce')
sess_tmp['exit_ts']  = pd.to_datetime(sess_tmp['exit'],  errors='coerce')
sess_tmp['calc_dur'] = (sess_tmp['exit_ts'] - sess_tmp['entry_ts']).dt.total_seconds() / 60

n_neg     = int((sess_tmp['calc_dur'] < 0).sum())
n_zero    = int((sess_tmp['calc_dur'] == 0).sum())
n_extreme = int((sess_tmp['calc_dur'] > 1440).sum())
n_miss_exit = int(sess_tmp['exit_ts'].isna().sum())

print(f'Missing exit timestamps  : {n_miss_exit}')
print(f'Missing duration values  : {int(raw_sessions["duration"].isna().sum())}')
print(f'Negative durations       : {n_neg}')
print(f'Zero durations           : {n_zero}')
print(f'Extreme (>24h) durations : {n_extreme}')
print(f'Exact dup rows (sessions): {int(raw_sessions.duplicated().sum())}')
print(f'Exact dup rows (payments): {int(raw_payments.duplicated().sum())}')
print(f'Neg payments             : {int((raw_payments["amount"] < 0).sum())}')

session_zones = set(raw_sessions['zone_id'].str.strip().str.upper())
master_zones  = set(raw_zones['zone_id'].str.strip().str.upper())
unresolved = session_zones - master_zones
print(f'Unresolved zone mismatches: {unresolved if unresolved else "None"}')

Missing exit timestamps  : 23
Missing duration values  : 10
Negative durations       : 8
Zero durations           : 0
Extreme (>24h) durations : 0
Exact dup rows (sessions): 12
Exact dup rows (payments): 5
Neg payments             : 0
Unresolved zone mismatches: None


In [7]:
dq_issues = [
    ('Missing exit timestamps',      'sessions', n_miss_exit,
     'Keep open sessions; exclude from duration calc & occupancy reconstruction',
     'Missing exit may be an active/open session'),
    ('Missing duration values',       'sessions', int(raw_sessions['duration'].isna().sum()),
     'Recalculate from entry/exit when both are valid', 'Timestamps are authoritative'),
    ('Negative calculated durations', 'sessions', n_neg,
     'Flag and exclude from duration/occupancy analyses', 'Entry after exit is logically invalid'),
    ('Zero durations',                'sessions', n_zero,
     'Exclude from duration analyses; retain for session count', 'Instantaneous sessions are implausible'),
    ('Extreme durations (>24 h)',     'sessions', n_extreme,
     'Flag and exclude from duration statistics', 'Treated as outliers'),
    ('Exact duplicate rows',          'sessions', int(raw_sessions.duplicated().sum()),
     'Remove all but first occurrence', 'Exact duplicates add no information'),
    ('Duplicate payment_id',          'payments', int(raw_payments['payment_id'].duplicated().sum()),
     'Remove duplicate payment rows', 'Duplicate IDs inflate fee totals'),
    ('Inconsistent zone labels',      'sessions', 0,
     'Normalise zone_id with strip/upper; all resolved to master', 'All zone labels match after standardisation'),
    ('Negative payment amounts',      'payments', int((raw_payments['amount'] < 0).sum()),
     'Flag; exclude from revenue totals', 'Negative fees are not valid parking charges'),
]
dq_df = pd.DataFrame(dq_issues, columns=['Issue','Dataset','Affected Records','Action','Reason'])
display(dq_df)
dq_df.to_csv(REPORTS_DIR / 'data_quality_report.csv', index=False)
dq_df.to_csv(PROJECT_ROOT / 'outputs' / 'data_quality_report.csv', index=False)
print('\n✓ Saved data_quality_report.csv')


,Issue,Dataset,Affected Records,Action,Reason
0,Missing exit timestamps,sessions,23,Keep open sessions; exclude from duration calc...,Missing exit may be an active/open session
1,Missing duration values,sessions,10,Recalculate from entry/exit when both are valid,Timestamps are authoritative
2,Negative calculated durations,sessions,8,Flag and exclude from duration/occupancy analyses,Entry after exit is logically invalid
3,Zero durations,sessions,0,Exclude from duration analyses; retain for ses...,Instantaneous sessions are implausible
4,Extreme durations (>24 h),sessions,0,Flag and exclude from duration statistics,Treated as outliers
5,Exact duplicate rows,sessions,12,Remove all but first occurrence,Exact duplicates add no information
6,Duplicate payment_id,payments,5,Remove duplicate payment rows,Duplicate IDs inflate fee totals
7,Inconsistent zone labels,sessions,0,Normalise zone_id with strip/upper; all resolv...,All zone labels match after standardisation
8,Negative payment amounts,payments,0,Flag; exclude from revenue totals,Negative fees are not valid parking charges



✓ Saved data_quality_report.csv


## 4  Data Cleaning Pipeline

In [8]:
sessions = raw_sessions.copy()
sessions['entry'] = pd.to_datetime(sessions['entry'], errors='coerce')
sessions['exit']  = pd.to_datetime(sessions['exit'],  errors='coerce')

n_before = len(sessions)
sessions = sessions.drop_duplicates()
print(f'After dedup: {len(sessions):,} rows (removed {n_before - len(sessions)})')

sessions['zone_id'] = sessions['zone_id'].str.strip().str.upper()
zones = raw_zones.copy()
zones['zone_id'] = zones['zone_id'].str.strip().str.upper()

sessions['calc_dur_min'] = (sessions['exit'] - sessions['entry']).dt.total_seconds() / 60
sessions['has_entry']    = sessions['entry'].notna()
sessions['has_exit']     = sessions['exit'].notna()
sessions['is_negative']  = sessions['calc_dur_min'] < 0
sessions['is_zero_dur']  = sessions['calc_dur_min'] == 0
sessions['is_extreme']   = sessions['calc_dur_min'] > 1440
sessions['is_open']      = sessions['has_entry'] & ~sessions['has_exit']
sessions['is_valid_dur'] = (sessions['has_entry'] & sessions['has_exit'] &
                             ~sessions['is_negative'] & ~sessions['is_zero_dur'] &
                             ~sessions['is_extreme'])
sessions['duration_min'] = np.where(sessions['is_valid_dur'], sessions['calc_dur_min'], np.nan)

print(f'Valid duration sessions  : {sessions["is_valid_dur"].sum():,}')
print(f'Open sessions            : {sessions["is_open"].sum():,}')
print(f'Negative duration        : {sessions["is_negative"].sum():,}')

payments = raw_payments.copy()
n_pay = len(payments)
payments = payments.drop_duplicates()
print(f'Payments after dedup: {len(payments):,} (removed {n_pay - len(payments)})')

calendar = raw_calendar.copy()
calendar['date'] = pd.to_datetime(calendar['date'], errors='coerce')
weather  = raw_weather.copy()
weather['date'] = pd.to_datetime(weather['date'], errors='coerce')

After dedup: 1,200 rows (removed 12)
Valid duration sessions  : 1,170
Open sessions            : 22
Negative duration        : 8
Payments after dedup: 1,200 (removed 5)


## 5  Dataset Integration

In [9]:
sessions['entry_date'] = sessions['entry'].dt.normalize()

df = sessions.merge(zones, on='zone_id', how='left')
print(f'After zones merge   : {len(df):,} rows')

df = df.merge(calendar.rename(columns={'date':'entry_date'}), on='entry_date', how='left')
print(f'After calendar merge: {len(df):,} rows')

df = df.merge(weather.rename(columns={'date':'entry_date'}), on='entry_date', how='left')
print(f'After weather merge : {len(df):,} rows')

df = df.merge(payments[['payment_id','payment_mode','amount']], on='payment_id', how='left')
print(f'After payments merge: {len(df):,} rows')

df.to_csv(DATA_PROC / 'cleaned_parking_data.csv', index=False)
print(f'\n✓ Saved cleaned_parking_data.csv -> {len(df):,} rows')
display(df.head(3))

After zones merge   : 1,200 rows
After calendar merge: 1,200 rows
After weather merge : 1,200 rows
After payments merge: 1,200 rows

✓ Saved cleaned_parking_data.csv -> 1,200 rows


,session_id,zone_id,zone,entry,exit,duration,fee,payment_id,calc_dur_min,has_entry,...,location,capacity,zone_type,weekday,holiday,event,rain,temperature,payment_mode,amount
0,S00001,Z03,Z03,2026-08-11 08:04:00,2026-08-11 07:34:00,52.0,26.0,P00001,-30.0,True,...,Riverfront,74,Recreational,Tuesday,0,0,0,35.2,Card,26.0
1,S00002,Z06,Z06,2026-08-31 10:02:00,2026-08-31 13:26:00,204.0,119.0,P00002,204.0,True,...,University,110,Educational,Monday,0,0,0,33.2,UPI,119.0
2,S00003,Z05,Z05,2026-09-20 16:14:00,2026-09-20 17:41:00,87.0,43.5,P00003,87.0,True,...,Market Street,95,Commercial,Sunday,1,0,0,26.9,Cash,43.5


## 6  Descriptive Statistics

In [10]:
valid = df[df['is_valid_dur']].copy()
n_days = (df['entry_date'].max() - df['entry_date'].min()).days + 1

print(f'Total sessions (after dedup) : {len(df):,}')
print(f'Valid-duration sessions      : {len(valid):,}')
print(f'Distinct zones               : {df["zone_id"].nunique()}')
print(f'Date range                   : {df["entry_date"].min().date()} to {df["entry_date"].max().date()} ({n_days} days)')
print(f'Avg duration (min)           : {valid["duration_min"].mean():.1f}')
print(f'Median duration (min)        : {valid["duration_min"].median():.1f}')
print(f'Total fees collected         : {df["amount"].sum():,.2f}')

zone_summary = df.groupby('zone_id').agg(
    sessions=('session_id','count'), valid_sessions=('is_valid_dur','sum'),
    avg_dur_min=('duration_min','mean'), median_dur_min=('duration_min','median'),
    total_fees=('amount','sum'), capacity=('capacity','first'),
    location=('location','first'), zone_type=('zone_type','first'),
).reset_index()
print('\nZone-level summary:')
display(zone_summary)

Total sessions (after dedup) : 1,200
Valid-duration sessions      : 1,170
Distinct zones               : 6
Date range                   : 2026-08-01 to 2026-09-29 (60 days)
Avg duration (min)           : 148.4
Median duration (min)        : 120.0
Total fees collected         : 80,882.44

Zone-level summary:


,zone_id,sessions,valid_sessions,avg_dur_min,median_dur_min,total_fees,capacity,location,zone_type
0,Z01,238,233,125.236052,103.0,13728.21,120,Central Plaza,Commercial
1,Z02,163,161,176.608696,154.0,12811.56,85,City Hospital,Hospital
2,Z03,147,139,157.179856,133.0,10491.14,74,Riverfront,Recreational
3,Z04,255,248,105.951613,87.5,11881.03,150,Railway Station,Transit
4,Z05,225,218,135.527523,116.0,14097.39,95,Market Street,Commercial
5,Z06,172,171,224.263158,180.0,17873.11,110,University,Educational


## 7  Occupancy Reconstruction

Occupancy is estimated using **entry/exit interval overlap** at 1-hour resolution,
restricted to **operating hours (7am–10pm)** to avoid diluting averages with empty-lot night hours.
Sessions without both a valid entry AND exit timestamp are excluded from this calculation.

> **Note:** This is a synthetic dataset with ~20 sessions/day across 6 zones.
> Peak simultaneous occupancy reaches ~6 vehicles. True occupancy % is therefore low
> relative to large zone capacities — this is a property of the data, not a calculation error.

In [11]:
valid_occ = valid[valid['entry'].notna() & valid['exit'].notna()].copy()
FREQ = '1h'
min_ts = valid_occ['entry'].min().floor(FREQ)
max_ts = valid_occ['exit'].max().ceil(FREQ)
time_index = pd.date_range(min_ts, max_ts, freq=FREQ)

# Operating hours only (7am-10pm) for meaningful occupancy analysis
time_index_ops = [t for t in time_index if 7 <= t.hour <= 22]

occ_records = []
for zone_id in valid_occ['zone_id'].unique():
    zs = valid_occ[valid_occ['zone_id'] == zone_id]
    cap_row = zones[zones['zone_id'] == zone_id]
    capacity = int(cap_row['capacity'].iloc[0]) if not cap_row.empty else np.nan
    for t in time_index_ops:
        t_end = t + pd.Timedelta(FREQ)
        count = int(((zs['entry'] < t_end) & (zs['exit'] >= t)).sum())
        rate  = (count / capacity * 100) if (pd.notna(capacity) and capacity > 0) else np.nan
        occ_records.append({'time_bucket': t, 'zone_id': zone_id,
                            'occupancy_count': count, 'capacity': capacity,
                            'occupancy_rate': rate})

occ_df = pd.DataFrame(occ_records)
occ_df = occ_df.merge(zones[['zone_id','location','zone_type']], on='zone_id', how='left')
occ_df['hour']         = occ_df['time_bucket'].dt.hour
occ_df['weekday_num']  = occ_df['time_bucket'].dt.weekday
occ_df['weekday_name'] = occ_df['time_bucket'].dt.day_name()
print(f'Occupancy grid (operating hours): {occ_df.shape}')

# Average and peak occupancy per zone
avg_occ = (occ_df[occ_df['occupancy_rate'].notna()]
           .groupby(['zone_id','location'])
           .agg(avg_occ_rate=('occupancy_rate','mean'),
                peak_occ_rate=('occupancy_rate','max'),
                avg_occ_count=('occupancy_count','mean'),
                peak_occ_count=('occupancy_count','max'))
           .reset_index()
           .sort_values('peak_occ_rate', ascending=False))
avg_occ['occupancy_rate']  = avg_occ['avg_occ_rate']
avg_occ['occupancy_count'] = avg_occ['avg_occ_count']

# Turnover calculation
turnover = (valid.groupby('zone_id')['session_id'].count().reset_index(name='valid_sessions')
                 .merge(zones[['zone_id','capacity','location']], on='zone_id', how='left'))
turnover['turnover_rate'] = (turnover['valid_sessions'] / turnover['capacity']).round(2)
turnover = turnover.sort_values('turnover_rate', ascending=False)

# Occupancy summary table
occ_summary = zone_summary.merge(avg_occ[['zone_id','avg_occ_count','peak_occ_count','avg_occ_rate','peak_occ_rate']],
                                 on='zone_id', how='left')
occ_summary = occ_summary.merge(turnover[['zone_id','turnover_rate']], on='zone_id', how='left')
occ_summary['avg_occ_rate'] = occ_summary['avg_occ_rate'].round(2)
occ_summary['peak_occ_rate'] = occ_summary['peak_occ_rate'].round(2)
occ_summary['avg_occ_count'] = occ_summary['avg_occ_count'].round(2)
occ_summary['avg_dur_min'] = occ_summary['avg_dur_min'].round(1)
occ_summary['median_dur_min'] = occ_summary['median_dur_min'].round(1)
occ_summary['total_fees'] = occ_summary['total_fees'].round(2)

occ_summary = occ_summary.rename(columns={
    'sessions': 'total_sessions',
    'avg_dur_min': 'avg_duration_min',
    'median_dur_min': 'median_duration_min',
    'total_fees': 'total_fees_collected',
    'avg_occ_rate': 'avg_occupancy_rate_pct',
    'peak_occ_rate': 'peak_occupancy_rate_pct',
})

summary_cols = ['zone_id','location','zone_type','capacity','total_sessions','valid_sessions',
                'turnover_rate','avg_occ_count','peak_occ_count','avg_occupancy_rate_pct',
                'peak_occupancy_rate_pct','avg_duration_min','median_duration_min','total_fees_collected']
occ_summary = occ_summary[summary_cols]
occ_summary.to_csv(PROJECT_ROOT / 'outputs' / 'occupancy_summary.csv', index=False)
occ_summary.to_csv(REPORTS_DIR / 'occupancy_summary.csv', index=False)
print('\n✓ Saved occupancy_summary.csv')
display(occ_summary)

# Enrich df for Excel analysis
df['entry_hour'] = df['entry'].dt.hour
df['time_period'] = df['entry_hour'].apply(assign_time_period)
df['is_weekend'] = df['entry'].dt.weekday.isin([5, 6]).astype(int)
df['day_type'] = np.where(df['is_weekend'] == 1, 'Weekend', 'Weekday')
df['event_label'] = np.where(df['event'] == 1, 'Event Day', 'Non-Event Day')
df['rain_label'] = np.where(df['rain'] == 1, 'Rainy Day', 'Non-Rainy Day')
df['duration_hours'] = (df['duration_min'] / 60).round(2)

zone_lookup = occ_summary[['zone_id','turnover_rate','avg_occ_count','peak_occ_count',
                           'avg_occupancy_rate_pct','peak_occupancy_rate_pct']].rename(
    columns={
        'turnover_rate': 'zone_turnover_rate',
        'avg_occ_count': 'zone_avg_occupancy',
        'peak_occ_count': 'zone_peak_occupancy',
        'avg_occupancy_rate_pct': 'zone_avg_occupancy_rate_pct',
        'peak_occupancy_rate_pct': 'zone_peak_occupancy_rate_pct',
    }
)
df = df.merge(zone_lookup, on='zone_id', how='left')
df.to_csv(DATA_PROC / 'cleaned_parking_data.csv', index=False)
print(f'✓ Saved enriched cleaned_parking_data.csv -> {len(df):,} rows x {len(df.columns)} cols')


Occupancy grid (operating hours): (5760, 10)

✓ Saved occupancy_summary.csv


,zone_id,location,zone_type,capacity,total_sessions,valid_sessions,turnover_rate,avg_occ_count,peak_occ_count,avg_occupancy_rate_pct,peak_occupancy_rate_pct,avg_duration_min,median_duration_min,total_fees_collected
0,Z01,Central Plaza,Commercial,120,238,233,1.94,0.70,4,0.58,3.33,125.2,103.0,13728.21
1,Z02,City Hospital,Hospital,85,163,161,1.89,0.63,6,0.74,7.06,176.6,154.0,12811.56
2,Z03,Riverfront,Recreational,74,147,139,1.88,0.49,4,0.66,5.41,157.2,133.0,10491.14
3,Z04,Railway Station,Transit,150,255,248,1.65,0.66,4,0.44,2.67,106.0,87.5,11881.03
4,Z05,Market Street,Commercial,95,225,218,2.29,0.69,6,0.72,6.32,135.5,116.0,14097.39
5,Z06,University,Educational,110,172,171,1.55,0.77,6,0.70,5.45,224.3,180.0,17873.11


✓ Saved enriched cleaned_parking_data.csv -> 1,200 rows x 40 cols


In [12]:
# Join calendar to occ_df for event analysis
cal_lookup = calendar.copy()
occ_df2 = occ_df.copy()
occ_df2['date_key'] = occ_df2['time_bucket'].dt.normalize()
occ_df2 = occ_df2.merge(cal_lookup.rename(columns={'date':'date_key'}), on='date_key', how='left')

# Turnover
n_days = (df['entry_date'].max() - df['entry_date'].min()).days + 1
turnover = (valid.groupby('zone_id')['session_id'].count().reset_index(name='valid_sessions')
                 .merge(zones[['zone_id','capacity','location']], on='zone_id', how='left'))
turnover['turnover_rate'] = turnover['valid_sessions'] / turnover['capacity']
turnover = turnover.sort_values('turnover_rate', ascending=False)

print('Turnover by zone:')
display(turnover)

Turnover by zone:


,zone_id,valid_sessions,capacity,location,turnover_rate
4,Z05,218,95,Market Street,2.294737
0,Z01,233,120,Central Plaza,1.941667
1,Z02,161,85,City Hospital,1.894118
2,Z03,139,74,Riverfront,1.878378
3,Z04,248,150,Railway Station,1.653333
5,Z06,171,110,University,1.554545


## 8  Exploratory Visualisations

In [13]:
# Chart 1 – Sessions by zone
sess_by_zone = (df.groupby(['zone_id','location'])['session_id'].count()
                  .reset_index(name='sessions')
                  .sort_values('sessions', ascending=False))
fig, ax = plt.subplots(figsize=(9, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')
bars = ax.bar(sess_by_zone['location'], sess_by_zone['sessions'],
              color=PALETTE[:len(sess_by_zone)], edgecolor='#0f1117')
for bar, val in zip(bars, sess_by_zone['sessions']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, str(val),
            ha='center', va='bottom', fontsize=9, color='#e0e0e0')
ax.set_title('Chart 1 — Parking Sessions by Zone', pad=12)
ax.set_xlabel('Zone / Location'); ax.set_ylabel('Number of Sessions')
ax.tick_params(axis='x', rotation=25); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); save_chart(fig, '01_sessions_by_zone')

  ✓ Saved 01_sessions_by_zone.png


**Interpretation:** Railway Station has the most sessions (255) followed by Central Plaza (238) and Market Street (225). Riverfront has the fewest (147). High session volume reflects demand but not necessarily high occupancy — a large zone can have many sessions at low simultaneous occupancy.

In [14]:
# Chart 2 – Peak occupancy rate by zone
fig, ax = plt.subplots(figsize=(9, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')
colors2 = [PALETTE[i % len(PALETTE)] for i in range(len(avg_occ))]
bars = ax.bar(avg_occ['location'], avg_occ['peak_occ_rate'],
              color=colors2, edgecolor='#0f1117')
for bar, val in zip(bars, avg_occ['peak_occ_rate']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{val:.1f}%',
            ha='center', va='bottom', fontsize=9, color='#e0e0e0')
ax.set_title('Chart 2 — Peak Occupancy Rate by Zone (Operating Hours, 7am-10pm)', pad=10)
ax.set_xlabel('Zone / Location'); ax.set_ylabel('Peak Occupancy Rate (%)')
ax.tick_params(axis='x', rotation=25); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); save_chart(fig, '02_occupancy_rate_by_zone')

  ✓ Saved 02_occupancy_rate_by_zone.png

**Interpretation:** City Hospital (7.1%) and Market Street (6.3%) have the highest peak occupancy rates. Railway Station (2.7%) has the lowest peak despite having the most sessions — this is because its large capacity (150 spaces) is rarely stressed simultaneously. All zones show low absolute occupancy due to the sparse synthetic dataset (~20 sessions/day across 6 zones).

In [15]:
# Chart 3 – Hourly occupancy
hourly_occ = occ_df.groupby('hour')['occupancy_count'].mean().reset_index(name='avg_occupancy')
fig, ax = plt.subplots(figsize=(11, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')
ax.fill_between(hourly_occ['hour'], hourly_occ['avg_occupancy'], alpha=0.25, color='#6c63ff')
ax.plot(hourly_occ['hour'], hourly_occ['avg_occupancy'], color='#6c63ff', linewidth=2.5, marker='o', markersize=5)
ax.set_title('Chart 3 — Average Hourly Parking Occupancy (All Zones, Operating Hours)', pad=12)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Avg Occupied Spaces')
ax.grid(alpha=0.3)
fig.tight_layout(); save_chart(fig, '03_hourly_occupancy')

  ✓ Saved 03_hourly_occupancy.png


**Interpretation:** Occupancy peaks around **16:00 (4 PM)**, reflecting afternoon demand. Activity is sustained from morning through evening, with a characteristic lunchtime and late-afternoon peak — consistent with commercial and transit zone usage patterns.

In [16]:
# Chart 4 – Heatmap weekday x hour
heatmap_data = (occ_df.groupby(['weekday_num','weekday_name','hour'])['occupancy_count']
                      .mean().reset_index())
heatmap_pivot = heatmap_data.pivot_table(index='weekday_num', columns='hour',
                                          values='occupancy_count', aggfunc='mean')
wd_labels = (heatmap_data.drop_duplicates('weekday_num')
                          .sort_values('weekday_num')['weekday_name'].tolist())
fig, ax = plt.subplots(figsize=(14, 5), facecolor='#0f1117')
ax.set_facecolor('#0f1117')
sns.heatmap(heatmap_pivot, ax=ax, cmap='magma', linewidths=0.3, linecolor='#0f1117',
            fmt='.0f', annot=True, annot_kws={'size': 7},
            yticklabels=wd_labels, cbar_kws={'label': 'Avg Occupied Spaces'})
ax.set_title('Chart 4 — Occupancy Heatmap: Weekday x Hour of Day', pad=12)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Day of Week')
ax.tick_params(colors='#e0e0e0')
fig.tight_layout(); save_chart(fig, '04_occupancy_heatmap')

  ✓ Saved 04_occupancy_heatmap.png


**Interpretation:** Thursday and Sunday show notable afternoon peaks (16–18h). Weekday mornings (9–11h) show consistent moderate occupancy. Saturday morning demand is also visible, aligned with retail/recreational activity.

In [17]:
# Chart 5 – Turnover by zone
fig, ax = plt.subplots(figsize=(9, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')
bars = ax.bar(turnover['location'], turnover['turnover_rate'],
              color=PALETTE[:len(turnover)], edgecolor='#0f1117')
for bar, val in zip(bars, turnover['turnover_rate']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.2f}',
            ha='center', va='bottom', fontsize=9, color='#e0e0e0')
ax.set_title('Chart 5 — Parking Turnover by Zone (Sessions per Space)', pad=10)
ax.set_xlabel('Zone / Location'); ax.set_ylabel('Turnover Rate (sessions / space)')
ax.tick_params(axis='x', rotation=25); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); save_chart(fig, '05_turnover_by_zone')

  ✓ Saved 05_turnover_by_zone.png


**Interpretation:** Market Street (2.29 sessions/space) and Central Plaza (1.94) have the highest turnover — spaces are reused frequently. University (1.55) has the lowest turnover, consistent with its long median duration (180 min) — students park for extended periods.

In [18]:
# Chart 6 – Event vs Non-event
event_occ = occ_df2.groupby('event')['occupancy_count'].agg(['mean','count']).reset_index()
event_occ['event_label'] = event_occ['event'].map({0:'Non-Event Day', 1:'Event Day'})

ev_vol = df.groupby('event')['session_id'].count().reset_index(name='sessions')
ev_vol['event_label'] = ev_vol['event'].map({0:'Non-Event Day', 1:'Event Day'})

fig, axes = plt.subplots(1, 2, figsize=(11, 5), facecolor='#0f1117')
for ax in axes: ax.set_facecolor('#1a1d27')
colors_ev = ['#4facfe','#f093fb']
axes[0].bar(event_occ['event_label'], event_occ['mean'], color=colors_ev, edgecolor='#0f1117')
for bar, val in zip(axes[0].patches, event_occ['mean']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.2f}',
                 ha='center', va='bottom', color='white', fontsize=10)
axes[0].set_title('Mean Occupied Spaces'); axes[0].set_ylabel('Avg Occupied Spaces'); axes[0].grid(axis='y', alpha=0.3)
axes[1].bar(ev_vol['event_label'], ev_vol['sessions'], color=colors_ev, edgecolor='#0f1117')
for bar, val in zip(axes[1].patches, ev_vol['sessions']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(val),
                 ha='center', va='bottom', color='white', fontsize=10)
axes[1].set_title('Total Session Count'); axes[1].set_ylabel('Sessions'); axes[1].grid(axis='y', alpha=0.3)
for ax in axes: ax.tick_params(axis='x', rotation=15)
fig.suptitle('Chart 6 — Event vs Non-Event Parking Behaviour', fontsize=14)
fig.tight_layout(); save_chart(fig, '06_event_vs_nonevent_occupancy')

display(event_occ); display(ev_vol)


  ✓ Saved 06_event_vs_nonevent_occupancy.png


,event,mean,count,event_label
0,0,0.657552,5376,Non-Event Day
1,1,0.656250,384,Event Day


,event,sessions,event_label
0,0,1123,Non-Event Day
1,1,77,Event Day


**Interpretation:** Event days show similar mean occupancy to non-event days (association only; confounders not controlled). The comparison is limited by a small number of event days (5 out of 60). Session volume is proportionally lower on event days, possibly due to event-specific routing or reduced regular weekday activity on those dates.

In [19]:
# Chart 7 – Rainy vs Non-rainy
rain_df = df.copy()
rain_df['rain_label'] = rain_df['rain'].map({0:'Non-Rainy Day', 1:'Rainy Day'})
rain_summary = rain_df.groupby('rain_label').agg(
    sessions=('session_id','count'), avg_dur_min=('duration_min','mean')).reset_index()

occ_df3w = occ_df2.copy()
occ_df3w['date_key2'] = occ_df3w['time_bucket'].dt.normalize()
occ_df3w = occ_df3w.merge(weather.rename(columns={'date':'date_key2'}), on='date_key2', how='left', suffixes=('','_wx'))
rain_col_w = 'rain_wx' if 'rain_wx' in occ_df3w.columns else 'rain'
occ_df3w['rain_label'] = occ_df3w[rain_col_w].map({0:'Non-Rainy Day', 1:'Rainy Day'})
rain_occ = occ_df3w.groupby('rain_label')['occupancy_count'].mean().reset_index(name='avg_occupancy')

fig, axes = plt.subplots(1, 3, figsize=(14, 5), facecolor='#0f1117')
for ax in axes: ax.set_facecolor('#1a1d27')
colors_r = ['#43e97b','#4facfe']
rain_s = rain_summary.sort_values('rain_label')
axes[0].bar(rain_s['rain_label'], rain_s['sessions'], color=colors_r, edgecolor='#0f1117')
axes[0].set_title('Session Volume'); axes[0].set_ylabel('Sessions'); axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(axes[0].patches, rain_s['sessions']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(val), ha='center', color='white', fontsize=10)
axes[1].bar(rain_s['rain_label'], rain_s['avg_dur_min'], color=colors_r, edgecolor='#0f1117')
axes[1].set_title('Avg Duration (min)'); axes[1].set_ylabel('Minutes'); axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(axes[1].patches, rain_s['avg_dur_min']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}', ha='center', color='white', fontsize=10)
rain_occ_s = rain_occ.sort_values('rain_label')
axes[2].bar(rain_occ_s['rain_label'], rain_occ_s['avg_occupancy'], color=colors_r, edgecolor='#0f1117')
axes[2].set_title('Avg Occupancy Count'); axes[2].set_ylabel('Occupied Spaces'); axes[2].grid(axis='y', alpha=0.3)
for bar, val in zip(axes[2].patches, rain_occ_s['avg_occupancy']):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.2f}', ha='center', color='white', fontsize=10)
for ax in axes: ax.tick_params(axis='x', rotation=15)
fig.suptitle('Chart 7 — Rainy vs Non-Rainy Parking Behaviour', fontsize=14)
fig.tight_layout(); save_chart(fig, '07_rainy_vs_nonrainy')

display(rain_summary); display(rain_occ)


  ✓ Saved 07_rainy_vs_nonrainy.png


,rain_label,sessions,avg_dur_min
0,Non-Rainy Day,992,148.687435
1,Rainy Day,208,147.082126


,rain_label,avg_occupancy
0,Non-Rainy Day,0.649583
1,Rainy Day,0.696875


**Interpretation:** Rainy days have slightly shorter average duration (147 vs 149 min) — a small difference that may not be meaningful given the limited number of rainy days (13/60). No strong weather-parking signal is evident in this dataset. This association-level finding requires caution before operational conclusions.

In [20]:
# Chart 8 – Duration by zone type
dur_df = valid[valid['zone_type'].notna()].copy()
zone_types = sorted(dur_df['zone_type'].unique())
colors_zt  = {zt: PALETTE[i % len(PALETTE)] for i, zt in enumerate(zone_types)}

fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor='#0f1117')
for ax in axes: ax.set_facecolor('#1a1d27')
bp_data = [dur_df[dur_df['zone_type']==zt]['duration_min'].dropna().values for zt in zone_types]
bp = axes[0].boxplot(bp_data, patch_artist=True, notch=False, medianprops={'color':'white','linewidth':2})
for patch, zt in zip(bp['boxes'], zone_types):
    patch.set_facecolor(colors_zt[zt]); patch.set_alpha(0.75)
axes[0].set_xticklabels(zone_types, rotation=20, fontsize=9)
axes[0].set_title('Box Plot — Duration by Zone Type')
axes[0].set_ylabel('Duration (minutes)'); axes[0].grid(axis='y', alpha=0.3)
sns.violinplot(data=dur_df, x='zone_type', y='duration_min', ax=axes[1],
               palette=list(colors_zt.values()), inner='quartile', cut=0)
axes[1].set_title('Violin Plot — Duration by Zone Type')
axes[1].set_xlabel('Zone Type'); axes[1].set_ylabel('Duration (minutes)')
axes[1].tick_params(axis='x', rotation=20); axes[1].grid(axis='y', alpha=0.3)
fig.suptitle('Chart 8 — Parking Duration Distribution by Zone Type', y=1.01)
fig.tight_layout(); save_chart(fig, '08_duration_by_zone_type')

print('Mean/Median duration by zone type:')
display(dur_df.groupby('zone_type')['duration_min'].agg(['count','mean','median','std']).round(1))

  ✓ Saved 08_duration_by_zone_type.png
Mean/Median duration by zone type:


,count,mean,median,std
zone_type,,,,
Commercial,451,130.2,108.0,86.8
Educational,171,224.3,180.0,149.8
Hospital,161,176.6,154.0,112.4
Recreational,139,157.2,133.0,96.5
Transit,248,106.0,87.5,71.5


**Interpretation:** Educational zones have the longest median stay (180 min — students park for lectures/labs). Transit zones are shortest (87.5 min — commuters park briefly). Commercial zones are in the middle (108 min). This confirms zone-type-specific maximum stay policies would be more effective than universal limits.

In [21]:
# Chart 9 – Weekday vs Weekend
occ_df_c = occ_df.copy()
occ_df_c['is_weekend'] = occ_df_c['weekday_num'].isin([5, 6])
wknd_occ = occ_df_c.groupby(['is_weekend','hour'])['occupancy_count'].mean().reset_index()
wknd_occ['day_type'] = wknd_occ['is_weekend'].map({False:'Weekday', True:'Weekend'})

fig, ax = plt.subplots(figsize=(11, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')
for dtype, color in [('Weekday','#6c63ff'),('Weekend','#ff6584')]:
    subset = wknd_occ[wknd_occ['day_type']==dtype]
    ax.plot(subset['hour'], subset['occupancy_count'], label=dtype, color=color, linewidth=2.5, marker='o', markersize=4)
    ax.fill_between(subset['hour'], subset['occupancy_count'], alpha=0.1, color=color)
ax.set_title('Chart 9 — Weekday vs Weekend Hourly Occupancy', pad=12)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Avg Occupied Spaces')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); save_chart(fig, '09_weekday_vs_weekend_occupancy')

  ✓ Saved 09_weekday_vs_weekend_occupancy.png


In [22]:
# Chart 10 – Fees by zone
fee_by_zone = (df.groupby(['zone_id','location'])['amount'].sum()
                 .reset_index(name='total_fees')
                 .sort_values('total_fees', ascending=False))
fig, ax = plt.subplots(figsize=(9, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')
bars = ax.bar(fee_by_zone['location'], fee_by_zone['total_fees'],
              color=PALETTE[:len(fee_by_zone)], edgecolor='#0f1117')
for bar, val in zip(bars, fee_by_zone['total_fees']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50, f'{val:,.0f}',
            ha='center', va='bottom', fontsize=9, color='white')
ax.set_title('Chart 10 — Total Parking Fees Collected by Zone', pad=12)
ax.set_xlabel('Zone / Location'); ax.set_ylabel('Total Fees (currency units)')
ax.tick_params(axis='x', rotation=25); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); save_chart(fig, '10_fees_by_zone')

  ✓ Saved 10_fees_by_zone.png


## 9  Answers to the Six Business Questions

In [23]:
top_zone = avg_occ.sort_values('peak_occ_rate', ascending=False).iloc[0]
bot_zone = avg_occ.sort_values('peak_occ_rate', ascending=False).iloc[-1]

print('Q1 — Highest occupancy zones (by peak operating-hour rate):')
display(avg_occ[['location','avg_occ_rate','peak_occ_rate','peak_occ_count']])

print(f'\n-> Top zone: {top_zone["location"]} (peak {top_zone["peak_occ_rate"]:.1f}%)')
print(f'-> Low zone: {bot_zone["location"]} (peak {bot_zone["peak_occ_rate"]:.1f}%)')

Q1 — Highest occupancy zones (by peak operating-hour rate):


,location,avg_occ_rate,peak_occ_rate,peak_occ_count
1,City Hospital,0.742647,7.058824,6
4,Market Street,0.723684,6.315789,6
5,University,0.699811,5.454545,6
2,Riverfront,0.664414,5.405405,4
0,Central Plaza,0.583333,3.333333,4
3,Railway Station,0.443056,2.666667,4



-> Top zone: City Hospital (peak 7.1%)
-> Low zone: Railway Station (peak 2.7%)


In [24]:
peak_hour = int(hourly_occ.loc[hourly_occ['avg_occupancy'].idxmax(), 'hour'])
print(f'Q2 — Peak parking hour: {peak_hour}:00')
print('Hourly occupancy (top 5):')
display(hourly_occ.sort_values('avg_occupancy', ascending=False).head())

Q2 — Peak parking hour: 16:00
Hourly occupancy (top 5):


,hour,avg_occupancy
9,16,0.894444
10,17,0.875000
8,15,0.838889
11,18,0.827778
7,14,0.800000


In [25]:
low_turn  = turnover.sort_values('turnover_rate').iloc[0]
high_turn = turnover.sort_values('turnover_rate', ascending=False).iloc[0]
print('Q3 — Turnover (ascending = lowest first):')
display(turnover[['location','capacity','valid_sessions','turnover_rate']])
print(f'\n-> Lowest turnover: {low_turn["location"]} ({low_turn["turnover_rate"]:.2f} sess/space)')

Q3 — Turnover (ascending = lowest first):


,location,capacity,valid_sessions,turnover_rate
4,Market Street,95,218,2.294737
0,Central Plaza,120,233,1.941667
1,City Hospital,85,161,1.894118
2,Riverfront,74,139,1.878378
3,Railway Station,150,248,1.653333
5,University,110,171,1.554545



-> Lowest turnover: University (1.55 sess/space)


In [26]:
print('Q4 — Event vs Non-Event occupancy:')
display(event_occ)
display(ev_vol)

Q4 — Event vs Non-Event occupancy:


,event,mean,count,event_label
0,0,0.657552,5376,Non-Event Day
1,1,0.656250,384,Event Day


,event,sessions,event_label
0,0,1123,Non-Event Day
1,1,77,Event Day


In [27]:
print('Q5 — Rainy vs Non-Rainy behaviour:')
display(rain_summary)
display(rain_occ)

Q5 — Rainy vs Non-Rainy behaviour:


,rain_label,sessions,avg_dur_min
0,Non-Rainy Day,992,148.687435
1,Rainy Day,208,147.082126


,rain_label,avg_occupancy
0,Non-Rainy Day,0.649583
1,Rainy Day,0.696875


In [28]:
print('Q6 — Duration by zone type:')
display(dur_df.groupby('zone_type')['duration_min'].agg(['count','mean','median','std']).round(1))

Q6 — Duration by zone type:


,count,mean,median,std
zone_type,,,,
Commercial,451,130.2,108.0,86.8
Educational,171,224.3,180.0,149.8
Hospital,161,176.6,154.0,112.4
Recreational,139,157.2,133.0,96.5
Transit,248,106.0,87.5,71.5


## 10  Feature Engineering

In [29]:
def assign_time_period(hour):
    if 0 <= hour < 6:   return 'Night'
    if 6 <= hour < 12:  return 'Morning'
    if 12 <= hour < 18: return 'Afternoon'
    return 'Evening'

def assign_util_cat(rate):
    """
    Utilization thresholds (Low<1%, Moderate 1-4%, High>=4%) are calibrated
    to this dataset's observed operating-hour rate range (0-8%).
    Standard thresholds (Low<50%, Moderate 50-85%, High>85%) apply to dense
    urban datasets with hundreds of sessions per hour.
    """
    if pd.isna(rate): return 'Unknown'
    if rate < 1.0:    return 'Low'
    if rate <= 4.0:   return 'Moderate'
    return 'High'

fe = df.copy()
fe['entry_hour']               = fe['entry'].dt.hour
fe['entry_weekday']            = fe['entry'].dt.weekday
fe['entry_month']              = fe['entry'].dt.month
fe['is_weekend']               = fe['entry_weekday'].isin([5, 6]).astype(int)
fe['time_period']              = fe['entry_hour'].apply(assign_time_period)
fe['session_duration_minutes'] = fe['duration_min']
fe['is_holiday']               = fe['holiday'].fillna(0).astype(int)
fe['is_event']                 = fe['event'].fillna(0).astype(int)
fe['is_rainy']                 = fe['rain'].fillna(0).astype(int)

zone_avg_occ = avg_occ[['zone_id','avg_occ_count','avg_occ_rate','peak_occ_rate']].rename(
    columns={'avg_occ_count':'avg_occ_count_z','avg_occ_rate':'avg_occ_rate_z','peak_occ_rate':'peak_occ_rate_z'})
fe = fe.merge(zone_avg_occ, on='zone_id', how='left')
fe['occupancy_count'] = fe['avg_occ_count_z']
fe['occupancy_rate']  = fe['avg_occ_rate_z']
fe['utilization_category'] = fe['occupancy_rate'].apply(assign_util_cat)
fe = fe.merge(turnover[['zone_id','turnover_rate']], on='zone_id', how='left')

fe_cols = ['session_id','zone_id','entry_date','entry_hour','entry_weekday','entry_month',
           'is_weekend','time_period','is_holiday','is_event','is_rainy','temperature',
           'zone_type','session_duration_minutes','occupancy_count','occupancy_rate',
           'utilization_category','turnover_rate','location','capacity','payment_mode','amount']
fe_save = fe[[c for c in fe_cols if c in fe.columns]].copy()
fe_save.to_csv(DATA_PROC / 'parking_features.csv', index=False)
print(f'Saved parking_features.csv: {len(fe_save):,} rows x {len(fe_save.columns)} cols')
print(f'Utilization categories: {fe_save["utilization_category"].value_counts().to_dict()}')
display(fe_save.head(3))

Saved parking_features.csv: 1,200 rows x 22 cols
Utilization categories: {'Low': 1200}


,session_id,zone_id,entry_date,entry_hour,entry_weekday,entry_month,is_weekend,time_period,is_holiday,is_event,...,zone_type,session_duration_minutes,occupancy_count,occupancy_rate,utilization_category,turnover_rate,location,capacity,payment_mode,amount
0,S00001,Z03,2026-08-11,8,1,8,0,Morning,0,0,...,Recreational,NaN,0.491667,0.664414,Low,1.878378,Riverfront,74,Card,26.0
1,S00002,Z06,2026-08-31,10,0,8,0,Morning,0,0,...,Educational,204.0,0.769792,0.699811,Low,1.554545,University,110,UPI,119.0
2,S00003,Z05,2026-09-20,16,6,9,1,Afternoon,1,0,...,Commercial,87.0,0.687500,0.723684,Low,2.294737,Market Street,95,Cash,43.5


In [30]:
feature_doc = [
    ('session_duration_minutes', 'Numeric', 'Duration of valid session in minutes'),
    ('entry_hour',               'Numeric', 'Hour of day session started (0-23)'),
    ('entry_weekday',            'Numeric', 'Day of week (0=Mon, 6=Sun)'),
    ('entry_month',              'Numeric', 'Month of session; seasonal patterns'),
    ('is_weekend',               'Binary',  '1 if session started Sat or Sun'),
    ('time_period',              'Nominal', 'Night/Morning/Afternoon/Evening'),
    ('is_holiday',               'Binary',  '1 if public holiday (from calendar.csv)'),
    ('is_event',                 'Binary',  '1 if city event occurred (from calendar.csv)'),
    ('is_rainy',                 'Binary',  '1 if it rained (from weather.csv)'),
    ('temperature',              'Numeric', 'Daily temperature (°C)'),
    ('zone_type',                'Nominal', 'Functional zone category (from zones.csv)'),
    ('occupancy_count',          'Numeric', 'Zone avg hourly occupied spaces (operating hours)'),
    ('occupancy_rate',           'Numeric', 'Zone avg hourly occupancy % (operating hours)'),
    ('utilization_category',     'Ordinal', 'Low/Moderate/High (thresholds: 1%, 4% for this dataset)'),
    ('turnover_rate',            'Numeric', 'Sessions per space over observation window'),
]
display(pd.DataFrame(feature_doc, columns=['Feature','Type','Description']))

,Feature,Type,Description
0,session_duration_minutes,Numeric,Duration of valid session in minutes
1,entry_hour,Numeric,Hour of day session started (0-23)
2,entry_weekday,Numeric,"Day of week (0=Mon, 6=Sun)"
3,entry_month,Numeric,Month of session; seasonal patterns
4,is_weekend,Binary,1 if session started Sat or Sun
5,time_period,Nominal,Night/Morning/Afternoon/Evening
6,is_holiday,Binary,1 if public holiday (from calendar.csv)
7,is_event,Binary,1 if city event occurred (from calendar.csv)
8,is_rainy,Binary,1 if it rained (from weather.csv)
9,temperature,Numeric,Daily temperature (°C)


## 11  Categorical Encoding

In [31]:
enc = fe_save.copy()
util_map = {'Low': 0, 'Moderate': 1, 'High': 2, 'Unknown': np.nan}
enc['utilization_category_enc'] = enc['utilization_category'].map(util_map)
nominal_cols = [c for c in ['zone_type','time_period','payment_mode'] if c in enc.columns]
enc_ohe = pd.get_dummies(enc, columns=nominal_cols, prefix=nominal_cols, drop_first=False, dtype=int)
print(f'Shape after encoding: {enc_ohe.shape}')
new_cols = [c for c in enc_ohe.columns if c not in enc.columns]
print(f'New encoded columns ({len(new_cols)}):', new_cols)
display(enc_ohe.head(3))

Shape after encoding: (1200, 33)
New encoded columns (13): ['zone_type_Commercial', 'zone_type_Educational', 'zone_type_Hospital', 'zone_type_Recreational', 'zone_type_Transit', 'time_period_Afternoon', 'time_period_Evening', 'time_period_Morning', 'time_period_Night', 'payment_mode_App', 'payment_mode_Card', 'payment_mode_Cash', 'payment_mode_UPI']


,session_id,zone_id,entry_date,entry_hour,entry_weekday,entry_month,is_weekend,is_holiday,is_event,is_rainy,...,zone_type_Recreational,zone_type_Transit,time_period_Afternoon,time_period_Evening,time_period_Morning,time_period_Night,payment_mode_App,payment_mode_Card,payment_mode_Cash,payment_mode_UPI
0,S00001,Z03,2026-08-11,8,1,8,0,0,0,0,...,1,0,0,0,1,0,0,1,0,0
1,S00002,Z06,2026-08-31,10,0,8,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
2,S00003,Z05,2026-09-20,16,6,9,1,1,0,0,...,0,0,1,0,0,0,0,0,1,0


**Encoding decisions:**
- `zone_type`, `time_period`, `payment_mode`: One-hot (nominal — no ordinal relationship)
- `utilization_category`: Label-encoded (Low=0, Moderate=1, High=2 — ordinal)
- Binary indicators already 0/1
- `entry_hour` kept numeric (0–23)

## 12  Feature Selection

In [32]:
# Target: session_duration_minutes (no future exit leakage)
drop_cols = ['session_id','entry_date','occupancy_count','occupancy_rate',
             'utilization_category','utilization_category_enc','location','amount']
feature_df = enc_ohe.drop(columns=[c for c in drop_cols if c in enc_ohe.columns], errors='ignore')
feature_df = feature_df.drop(columns=['zone_id'], errors='ignore')
target_col = 'session_duration_minutes'
modelling_df = feature_df[feature_df[target_col].notna()].copy()
non_numeric = modelling_df.select_dtypes(exclude='number').columns.tolist()
if non_numeric:
    modelling_df = modelling_df.drop(columns=non_numeric)
X = modelling_df.drop(columns=[target_col]).fillna(modelling_df.median(numeric_only=True))
y = modelling_df[target_col]
const_cols = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=const_cols)
print(f'Feature matrix: {X.shape}')

Feature matrix: (1170, 23)


In [33]:
# Correlation matrix
corr = X.select_dtypes(include='number').corr()
fig, ax = plt.subplots(figsize=(max(8, len(corr)), max(6, len(corr))), facecolor='#0f1117')
ax.set_facecolor('#0f1117')
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap='coolwarm', center=0, annot=True,
            fmt='.2f', annot_kws={'size': 7}, linewidths=0.3,
            linecolor='#0f1117', cbar_kws={'label': 'Pearson r'})
ax.set_title('Feature Correlation Matrix', pad=10)
ax.tick_params(colors='#e0e0e0', labelsize=8)
fig.tight_layout(); save_chart(fig, '11_correlation_matrix')

  ✓ Saved 11_correlation_matrix.png


In [34]:
# Mutual Information
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = pd.DataFrame({'feature': X.columns, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 6), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')
colors_mi = plt.cm.viridis(np.linspace(0.2, 0.9, len(mi_df)))
ax.barh(mi_df['feature'][::-1], mi_df['mi_score'][::-1], color=colors_mi)
ax.set_title('Mutual Information Scores — Target: session_duration_minutes', pad=10)
ax.set_xlabel('Mutual Information Score'); ax.grid(axis='x', alpha=0.3)
fig.tight_layout(); save_chart(fig, '12_mutual_information')

print('Top 10 features by MI:')
display(mi_df.head(10))

  ✓ Saved 12_mutual_information.png
Top 10 features by MI:


,feature,mi_score
0,capacity,0.077184
1,turnover_rate,0.067158
2,zone_type_Transit,0.032374
3,zone_type_Educational,0.028911
4,payment_mode_Cash,0.021020
5,zone_type_Recreational,0.020908
6,temperature,0.017180
7,payment_mode_App,0.013202
8,zone_type_Hospital,0.011351
9,time_period_Morning,0.010329


In [35]:
selected = mi_df[mi_df['mi_score'] > 0].copy()
rationale_map = {
    'capacity':'Larger zones attract different stay lengths',
    'turnover_rate':'High-turnover zones have shorter sessions',
    'entry_hour':'Time-of-day drives duration',
    'is_rainy':'Weather may affect how long drivers park',
    'is_event':'Events may change parking behaviour',
    'temperature':'Extreme temperatures alter comfort and dwell time',
}
selected['rationale'] = selected['feature'].map(rationale_map).fillna('Identified by MI analysis')
ohe_sel = [c for c in X.columns if any(c.startswith(p) for p in ['zone_type_','time_period_','payment_mode_'])]
ohe_rows = pd.DataFrame({'feature': ohe_sel,
    'mi_score': [float(mi_df.set_index('feature').loc[c,'mi_score']) if c in mi_df['feature'].values else 0 for c in ohe_sel],
    'rationale': ['One-hot encoded nominal categorical']*len(ohe_sel)})
final_sel = pd.concat([selected, ohe_rows], ignore_index=True).drop_duplicates('feature')
final_sel.to_csv(PROJECT_ROOT / 'outputs' / 'selected_features.csv', index=False)
print(f'Saved selected_features.csv: {len(final_sel)} features')
display(final_sel)

Saved selected_features.csv: 17 features


,feature,mi_score,rationale
0,capacity,0.077184,Larger zones attract different stay lengths
1,turnover_rate,0.067158,High-turnover zones have shorter sessions
2,zone_type_Transit,0.032374,Identified by MI analysis
3,zone_type_Educational,0.028911,Identified by MI analysis
4,payment_mode_Cash,0.021020,Identified by MI analysis
5,zone_type_Recreational,0.020908,Identified by MI analysis
6,temperature,0.017180,Extreme temperatures alter comfort and dwell time
7,payment_mode_App,0.013202,Identified by MI analysis
8,zone_type_Hospital,0.011351,Identified by MI analysis
9,time_period_Morning,0.010329,Identified by MI analysis


## 13  Limitations

| Limitation | Impact | Mitigation |
|------------|--------|------------|
| Synthetic practice data | Patterns may not represent true urban parking | Methods are valid; findings illustrative |
| Open sessions excluded from occupancy | Slight undercount | Flagged; session-volume metric provided separately |
| Negative-duration sessions excluded | Small data loss | Documented; sign flip not applied |
| Zone-average occupancy assigned to sessions | Temporal approximation | Time-resolved occ_df available |
| Sparse data (~20 sessions/day) | Low absolute occupancy % | Noted in all Q1 findings |
| Short 60-day window | Seasonal limits | Acknowledged in recommendations |
| Observational only | No causal claims | Stated explicitly |

## 14  Final Findings & Business Recommendations

In [36]:
rain_dur_data = rain_df.groupby('rain')['duration_min'].mean()
rain_dur   = float(rain_dur_data.loc[1]) if 1 in rain_dur_data.index else float('nan')
norain_dur = float(rain_dur_data.loc[0]) if 0 in rain_dur_data.index else float('nan')

ev_mean_val  = event_occ[event_occ['event']==1]['mean'].values
non_ev_val   = event_occ[event_occ['event']==0]['mean'].values
ev_mean_str  = f'{ev_mean_val[0]:.2f}' if len(ev_mean_val) else 'N/A'
non_ev_str   = f'{non_ev_val[0]:.2f}'  if len(non_ev_val)  else 'N/A'

dur_by_type = dur_df.groupby('zone_type')['duration_min'].agg(['mean','median']).round(1)

findings = [
    ('Q1 — Highest occupancy',
     f'Peak occ: {avg_occ[["location","peak_occ_rate"]].set_index("location")["peak_occ_rate"].round(1).to_dict()}',
     f'{top_zone["location"]} highest peak ({top_zone["peak_occ_rate"]:.1f}%); {bot_zone["location"]} lowest ({bot_zone["peak_occ_rate"]:.1f}%)',
     f'Prioritise enforcement at City Hospital & Market Street; review pricing at Railway Station'),
    ('Q2 — Peak periods',
     f'{len(valid_occ):,} valid sessions; peak hour computed from 1h occupancy buckets',
     f'Peak demand at {peak_hour}:00; weekday/weekend profiles differ (Charts 3,4,9)',
     'Staff and signage peak from 15:00-17:00; consider time-based pricing'),
    ('Q3 — Low turnover',
     f'Turnover range {turnover["turnover_rate"].min():.2f}-{turnover["turnover_rate"].max():.2f}',
     f'{low_turn["location"]} lowest turnover ({low_turn["turnover_rate"]:.2f} sess/space)',
     'Introduce max-stay limits or escalating tariffs in low-turnover zones'),
    ('Q4 — Event effect',
     f'Event days: mean occ={ev_mean_str} vs Non-event={non_ev_str} (association only)',
     'Event days show similar occupancy to non-event days; difference not substantial in this dataset',
     'Continue monitoring; larger dataset needed before event-surge pricing decisions'),
    ('Q5 — Rain effect',
     f'Rain avg dur={rain_dur:.1f} min; Non-rain={norain_dur:.1f} min',
     f'{"Rainy days shorter" if rain_dur < norain_dur else "Rainy days longer"} by {abs(rain_dur-norain_dur):.1f} min (association; confounders not controlled)',
     'Collect more data before weather-based pricing; monitor rainy-day patterns'),
    ('Q6 — Duration by zone type',
     f'{dur_by_type.to_dict()}',
     f'Educational longest (median 180 min); Transit shortest (87.5 min)',
     'Tailor stay limits per zone type: short limits for Transit, longer for Educational & Hospital'),
]
find_df = pd.DataFrame(findings, columns=['Business Question','Evidence','Key Finding','Recommended Action'])
display(find_df)
find_df.to_csv(REPORTS_DIR / 'business_findings.csv', index=False)
find_df.to_csv(PROJECT_ROOT / 'outputs' / 'business_findings.csv', index=False)
print('\n✓ Saved business_findings.csv')


,Business Question,Evidence,Key Finding,Recommended Action
0,Q1 — Highest occupancy,"Peak occ: {'City Hospital': 7.1, 'Market Stree...",City Hospital highest peak (7.1%); Railway Sta...,Prioritise enforcement at City Hospital & Mark...
1,Q2 — Peak periods,"1,170 valid sessions; peak hour computed from ...",Peak demand at 16:00; weekday/weekend profiles...,Staff and signage peak from 15:00-17:00; consi...
2,Q3 — Low turnover,Turnover range 1.55-2.29,University lowest turnover (1.55 sess/space),Introduce max-stay limits or escalating tariff...
3,Q4 — Event effect,Event days: mean occ=0.66 vs Non-event=0.66 (a...,Event days show similar occupancy to non-event...,Continue monitoring; larger dataset needed bef...
4,Q5 — Rain effect,Rain avg dur=147.1 min; Non-rain=148.7 min,Rainy days shorter by 1.6 min (association; co...,Collect more data before weather-based pricing...
5,Q6 — Duration by zone type,"{'mean': {'Commercial': 130.2, 'Educational': ...",Educational longest (median 180 min); Transit ...,Tailor stay limits per zone type: short limits...



✓ Saved business_findings.csv


In [37]:
# Final deliverable check
files_to_check = [
    DATA_PROC / 'cleaned_parking_data.csv',
    DATA_PROC / 'parking_features.csv',
    PROJECT_ROOT / 'outputs' / 'selected_features.csv',
    PROJECT_ROOT / 'outputs' / 'data_quality_report.csv',
    PROJECT_ROOT / 'outputs' / 'occupancy_summary.csv',
    PROJECT_ROOT / 'outputs' / 'business_findings.csv',
    REPORTS_DIR / 'data_quality_report.csv',
    REPORTS_DIR / 'business_findings.csv',
    REPORTS_DIR / 'occupancy_summary.csv',
]
print('=== Deliverable Checklist ===')
for fp in files_to_check:
    status = 'PASS' if fp.exists() else 'FAIL'
    print(f'  [{status}] {fp.relative_to(PROJECT_ROOT)}')

chart_files = sorted(CHARTS_DIR.glob('*.png'))
print(f'\n  Charts: {len(chart_files)}')
for cp in chart_files:
    print(f'    [PASS] {cp.name}')

print('\n=== Analysis Complete! ===')


=== Deliverable Checklist ===
  [PASS] data/processed/cleaned_parking_data.csv
  [PASS] data/processed/parking_features.csv
  [PASS] outputs/selected_features.csv
  [PASS] outputs/data_quality_report.csv
  [PASS] outputs/occupancy_summary.csv
  [PASS] outputs/business_findings.csv
  [PASS] outputs/reports/data_quality_report.csv
  [PASS] outputs/reports/business_findings.csv
  [PASS] outputs/reports/occupancy_summary.csv

  Charts: 12
    [PASS] 01_sessions_by_zone.png
    [PASS] 02_occupancy_rate_by_zone.png
    [PASS] 03_hourly_occupancy.png
    [PASS] 04_occupancy_heatmap.png
    [PASS] 05_turnover_by_zone.png
    [PASS] 06_event_vs_nonevent_occupancy.png
    [PASS] 07_rainy_vs_nonrainy.png
    [PASS] 08_duration_by_zone_type.png
    [PASS] 09_weekday_vs_weekend_occupancy.png
    [PASS] 10_fees_by_zone.png
    [PASS] 11_correlation_matrix.png
    [PASS] 12_mutual_information.png

=== Analysis Complete! ===
